# Stream models: training on a GPU runtime

Trains small sequence models on the per-field band medians (the control) and, on the fields that have
pixel chips, the models that read the pixels through the season. Every setting is fixed in the code;
nothing is tuned here. Upload the files listed in `README.md` into `/content`, pick a GPU runtime
(T4 is enough; high RAM if offered), then run the cells top to bottom. Every training step can be
rerun: finished models are skipped. Step 4 needs no chips and can run before the extraction job is done.


In [ ]:
# 1. Settings
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
UPLOAD, WORK, OUT = Path("/content"), Path("/content/work"), Path("/content/out")
bundle = json.loads((UPLOAD / "bundle.json").read_text())
RUNS = bundle["runs"]                     # runs whose chips are used; all must be present for steps 5-8
SEEDS = 3
need = [bundle["code_zip"], "folds_full.parquet", "folds_subset.parquet", *[f"ts_{r}.parquet" for r in RUNS]]
print("missing uploads:", [n for n in need if not (UPLOAD / n).exists()] or "none")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU: change the runtime type")
print("cpus", os.cpu_count())


In [ ]:
# 2. Optional: Google Drive (chip zips are read from MyDrive/fieldvision_stream_px/ when they are not
#    in /content, and results are copied to MyDrive/fieldvision_stream_train/)
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Unpack the code, check it, print versions
CODE = WORK / "code"
shutil.rmtree(CODE, ignore_errors=True)
assert hashlib.sha256((UPLOAD / bundle["code_zip"]).read_bytes()).hexdigest() == bundle["code_sha256"], "code zip differs"
zipfile.ZipFile(UPLOAD / bundle["code_zip"]).extractall(CODE)
env = dict(os.environ, PYTHONPATH=str(CODE))
OUT.mkdir(parents=True, exist_ok=True)
versions = subprocess.run([sys.executable, "-c", "import torch, numpy, pandas, sklearn, sys; print('python', sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'numpy', numpy.__version__, 'pandas', pandas.__version__, 'sklearn', sklearn.__version__)"],
                          capture_output=True, text=True).stdout.strip()
print(versions)
DRIVE = Path("/content/drive/MyDrive")

def train(name, folds, timeseries, models, times, schemes, chips=()):
    """One training process; returns it so several can run side by side."""
    cmd = [sys.executable, "-W", "ignore", "-m", "fieldvision.stream.train", "--folds", str(UPLOAD / folds),
           "--timeseries", *map(str, timeseries), "--models", *models, "--time", *times, "--schemes", *schemes,
           "--seeds", str(SEEDS), "--threads", "2", "--out", str(OUT / name)]
    if chips:
        cmd += ["--chips", *map(str, chips)]
    path = OUT / f"{name}.log"
    START[name] = len(path.read_text().splitlines()) if path.exists() else 0
    return name, subprocess.Popen(cmd, env=env, stdout=open(path, "a"), stderr=subprocess.STDOUT, text=True)

START = {}

def wait(procs):
    """Print each process's new result lines until all have ended; stop the cell if one failed."""
    seen = {name: START.get(name, 0) for name, _ in procs}
    while True:
        alive = [p.poll() is None for _, p in procs]
        for name, _ in procs:
            lines = (OUT / f"{name}.log").read_text().splitlines()
            for line in lines[seen[name]:]:
                if "ROC-AUC" in line or "fields" in line or "Error" in line or "wrote" in line:
                    print(f"[{time.strftime('%H:%M:%S')}] {name} | {line}", flush=True)
            seen[name] = len(lines)
        if not any(alive):
            break
        time.sleep(20)
    bad = [name for name, p in procs if p.returncode != 0]
    if bad:
        for name in bad:
            print((OUT / f"{name}.log").read_text()[-3000:])
        raise SystemExit(f"failed: {bad} (rerun the cell to resume)")
    if DRIVE.exists():
        shutil.copytree(OUT, DRIVE / "fieldvision_stream_train", dirs_exist_ok=True)


In [ ]:
# 4. Control on ALL labelled European fields (band medians only, no chips): each country held out, then
#    the spatial folds. About 15 minutes on a T4.
ts_full = [UPLOAD / f"ts_{r}.parquet" for r in bundle["all_runs"]]
wait([train("control_full", "folds_full.parquet", ts_full, ["seq"], ["greenup", "calendar"], ["loco", "spatial"])])


In [ ]:
# 5. Chips: find each run's parts (in /content, else on Drive), verify, join, unpack
PX = WORK / "px"
for run in RUNS:
    if (PX / run / "chips.npz").exists():
        continue
    tag = f"stream_px_{run}"
    src = next((d for d in (UPLOAD, DRIVE / "fieldvision_stream_px") if (d / f"{tag}_SHA256SUMS.txt").exists()), None)
    assert src is not None, f"{tag}_SHA256SUMS.txt not found in /content or on Drive"
    parts = []
    for line in (src / f"{tag}_SHA256SUMS.txt").read_text().splitlines():
        digest, name = line.split()
        if name[-4] == "." and name[-3:].isdigit():
            data = (src / name).read_bytes()
            assert hashlib.sha256(data).hexdigest() == digest, f"{name}: checksum differs, upload it again"
            parts.append((name, data))
    blob = b"".join(d for _, d in sorted(parts))
    (PX / run).mkdir(parents=True, exist_ok=True)
    (WORK / f"{tag}.zip").write_bytes(blob)
    with zipfile.ZipFile(WORK / f"{tag}.zip") as z:
        assert z.testzip() is None
        z.extractall(PX / run)
    m = json.loads((PX / run / "manifest.json").read_text())
    print(run, m.get("status"), "fields with chips", m.get("fields_with_chips"), "chips", m.get("n_obs"),
          f"{(PX / run / 'chips.npz').stat().st_size / 1e6:.0f} MB")
chips = [PX / r / "chips.npz" for r in RUNS]
ts_sub = [PX / r / "timeseries.parquet" for r in RUNS]


In [ ]:
# 6. Stage A (the result that matters): fields with chips, each country held out in turn.
#    Four processes side by side; the control (seq) is retrained on exactly these fields.
def stage(schemes, chip_times):
    return [train("subset_seq", "folds_subset.parquet", ts_sub, ["seq"], ["greenup", "calendar"], schemes, chips),
            train("subset_pixelset", "folds_subset.parquet", ts_sub, ["pixelset"], chip_times, schemes, chips),
            train("subset_chipseq", "folds_subset.parquet", ts_sub, ["chipseq"], chip_times, schemes, chips),
            train("subset_conv3d", "folds_subset.parquet", ts_sub, ["conv3d"], chip_times, schemes, chips)]
wait(stage(["loco"], ["greenup"]))


In [ ]:
# 7. Stage B: the same models on the spatial folds (gives the in-country number for the loss column)
wait(stage(["loco", "spatial"], ["greenup"]))


In [ ]:
# 8. Stage C (only if there is time): the chip models with calendar days instead of days since green-up
wait(stage(["loco", "spatial"], ["greenup", "calendar"]))


In [ ]:
# 9. Results so far, as one zip (run after step 4, 6, 7 or 8; the name says which stages are in it).
#    Fetch the zip and the checksum file from the file panel.
done = [d.name for d in sorted(OUT.iterdir()) if d.is_dir() and (d / "predictions.parquet").exists()]
import pandas as pd
frames = {d: pd.read_parquet(OUT / d / "predictions.parquet") for d in done}
schemes = sorted({s.split(":")[0] for f in frames.values() for s in f["scheme"].unique()})
models = sorted({m for f in frames.values() for m in f["model"].unique()})
stage_tag = ("0" if "control_full" in done else "") + ("A" if any(d.startswith("subset") for d in done) else "") \
    + ("B" if any(d.startswith("subset") and "spatial" in set(frames[d]["scheme"]) for d in done) else "") \
    + ("C" if any(m.endswith("-cal") and not m.startswith("seq") for m in models) else "")
result = Path(f"/content/stream_train_{stage_tag}_colab_result.zip")
with zipfile.ZipFile(result, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for d in done:
        z.write(OUT / d / "predictions.parquet", f"{d}/predictions.parquet")
        z.write(OUT / d / "settings.json", f"{d}/settings.json")
        z.write(OUT / f"{d}.log", f"{d}.log")
    z.writestr("colab_environment.txt", versions + "\n")
sums = Path(f"/content/stream_train_{stage_tag}_SHA256SUMS.txt")
sums.write_text(f"{hashlib.sha256(result.read_bytes()).hexdigest()}  {result.name}\n")
print(result.name, result.stat().st_size // 1024, "KB;", sums.name)
print("folders:", done, "| schemes:", schemes, "| models:", models)
if DRIVE.exists():
    (DRIVE / "fieldvision_stream_train").mkdir(exist_ok=True)
    shutil.copy2(result, DRIVE / "fieldvision_stream_train"); shutil.copy2(sums, DRIVE / "fieldvision_stream_train")
